In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import gradio as gr
from datasets import load_dataset
import sacrebleu
import speech_recognition as sr
from typing import List, Tuple, Optional
import yaml
import warnings
warnings.filterwarnings('ignore')

W0329 20:56:46.814000 25028 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
print(f"GPU name: {torch.cuda.get_device_name(0)}")

GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
print(f"GPU properties: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

GPU properties: 6.44 GB


In [5]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

# Use config values
print(config["messages"]["loading"])
print(config["messages"]["waiting"])

Loading model and tokenizer...
Please wait while the model is being loaded. This may take a few moments.


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    config["model"]["name"],
    src_lang=config["model"]["src_lang"],
    use_fast=config["model"]["use_fast_tokenizer"]
)

In [7]:
MODEL_NAME = config["model"]["name"]

In [8]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

In [9]:
print("Model loaded successfully!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Model dtype: {model.dtype}")

Model loaded successfully!
Model parameters: 615.1M
Model dtype: torch.float16


In [10]:
# Language code mapping (FLORES-200 format)
LANGUAGE_CODES = {
    "English": "eng_Latn",
    "French": "fra_Latn",
    "Arabic": "arb_Arab",
    "Spanish": "spa_Latn",
    "German": "deu_Latn",
    "Chinese (Simplified)": "zho_Hans",
    "Chinese (Traditional)": "zho_Hant",
    "Japanese": "jpn_Jpan",
    "Korean": "kor_Hang",
    "Russian": "rus_Cyrl",
    "Portuguese": "por_Latn",
    "Italian": "ita_Latn",
    "Dutch": "nld_Latn",
    "Turkish": "tur_Latn",
    "Polish": "pol_Latn",
    "Hindi": "hin_Deva",
    "Bengali": "ben_Beng",
    "Urdu": "urd_Arab",
    "Vietnamese": "vie_Latn",
    "Thai": "tha_Thai",
    "Indonesian": "ind_Latn",
    "Malay": "zsm_Latn",
    "Swahili": "swh_Latn",
    "Greek": "ell_Grek",
    "Hebrew": "heb_Hebr",
    "Persian": "pes_Arab",
    "Ukrainian": "ukr_Cyrl",
    "Czech": "ces_Latn",
    "Swedish": "swe_Latn",
    "Danish": "dan_Latn",
    "Finnish": "fin_Latn",
    "Norwegian": "nob_Latn",
    "Hungarian": "hun_Latn",
    "Romanian": "ron_Latn",
    "Bulgarian": "bul_Cyrl",
    "Croatian": "hrv_Latn",
    "Serbian": "srp_Cyrl",
    "Slovak": "slk_Latn",
    "Lithuanian": "lit_Latn",
    "Latvian": "lvs_Latn",
    "Estonian": "est_Latn",
    "Slovenian": "slv_Latn",
    "Catalan": "cat_Latn",
    "Tagalog": "tgl_Latn",
    "Tamil": "tam_Taml",
    "Telugu": "tel_Telu",
    "Kannada": "kan_Knda",
    "Malayalam": "mal_Mlym",
    "Marathi": "mar_Deva",
    "Gujarati": "guj_Gujr"
}

# Get list of supported languages
SUPPORTED_LANGUAGES = list(LANGUAGE_CODES.keys())

print(f"Configured {len(SUPPORTED_LANGUAGES)} languages")
print(f"\nSample languages: {', '.join(SUPPORTED_LANGUAGES[:10])}...")

Configured 50 languages

Sample languages: English, French, Arabic, Spanish, German, Chinese (Simplified), Chinese (Traditional), Japanese, Korean, Russian...
